In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt


X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)




In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)




In [ ]:
# 3. Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)




In [ ]:
# 4. Print shape of one batch
data_iter = iter(train_loader)
images, labels = next(data_iter)
print(f"Batch images shape: {images.shape}") # Should be [32, 3, H, W]
print(f"Batch labels shape: {labels.shape}")


In [ ]:
# 5. Display sample images
plt.figure(figsize=(10, 5))
for i in range(4):
    plt.subplot(1, 4, i+1)
    # Permute back to (H, W, C) for matplotlib
    plt.imshow(images[i].permute(1, 2, 0).numpy())
    plt.title(f"Age: {labels[i].item()}")
    plt.axis('off')
plt.show()


In [ ]:
import torch.nn as nn
import torch.optim as optim


In [ ]:
# Task 1: Write your model class here:a# Task 1: Model Class (Flattening images for Linear Layers)
class AgePredictor(nn.Module):
    def __init__(self, input_shape):
        super(AgePredictor, self).__init__()
        # input_shape is (C, H, W)
        self.flatten = nn.Flatten()
        in_features = input_shape[0] * input_shape[1] * input_shape[2]

        self.network = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1) # Single output for regression
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)



In [ ]:
# Task 2 & 3: Training and Validation Loops
def train_step(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(loader)

def val_step(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
    return running_loss / len(loader)


In [ ]:

# Task 4: Define device, model, loss, optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_sample_shape = X_train.shape[1:]
model = AgePredictor(input_sample_shape).to(device)
criterion = nn.MSELoss() # Mean Squared Error for regression
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# Task 5: Start training
train_losses = []
val_losses = []

for epoch in range(20):
    t_loss = train_step(model, train_loader, criterion, optimizer, device)
    v_loss = val_step(model, test_loader, criterion, device)
    train_losses.append(t_loss)
    val_losses.append(v_loss)
    print(f"Epoch {epoch+1}/20 | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f}")

In [ ]:
# Task 1: Plot Losses
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()



In [ ]:
# Task 2 (Bonus): Predictions vs Actual
model.eval()
with torch.no_grad():
    images, labels = next(iter(test_loader))
    images = images.to(device)
    preds = model(images).cpu().numpy()

plt.figure(figsize=(12, 6))
for i in range(4):
    plt.subplot(1, 4, i+1)
    plt.imshow(images[i].cpu().permute(1, 2, 0).numpy())
    plt.title(f"Act: {labels[i].item():.1f}\nPred: {preds[i][0]:.1f}")
    plt.axis('off')
plt.show()